In [1]:
# -*- coding: utf-8 -*-
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import backtest

In [2]:
#plt.rcParams['font.sans-serif'] = ['SimHei']
#plt.rcParams['axes.unicode_minus'] = False

# =================配置区域=================
SIGNAL_PATH = r"C:/Users/jltao/Desktop/timing/signal"
INDEX_CLOSE_PATH = "//eqserver/data/trade/index_close.pkl"
INDEX_OPEN_PATH = "//eqserver/data/trade/index_open.pkl"
OUTPUT_PATH = r"C:/Users/jltao/Desktop/timing/signal"


signal_file_names = [
                    "IC_basis_signal",
                    "pcr_signal",
                    "skew_signal",
                    "spread_signal",
                    "usdcnh_signal",
                    "vol_signal",
                    "basis_signal",
                    "highlow_signal",
                    "std_signal",
                    "idy_turn_pct_signal",
                    "trend_signal",
                    "idy_amt_corr_signal",
                    "sc_amt_pct_signal",
                    "amt_signal",
                    "CDS_signal",
                    "margin_signal",
                    "AIAE_signal",
                    "monetarypolicy_signal",
                    "pb_signal",
                    "pe_signal",
                    "monetarycondition_signal",
                    "inflation_signal",
                    "growth_signal",
                    "ht_signal",
                    "macro_cat_signal",
                    "value_cat_signal",
                    "option_cat_signal",
                    "moneyflow_cat_signal",
                    "price_cat_signal",
                    "cmb_signal",
                    ]

ENET_CONFIG = {
    'groups': {
        "macro": ["growth", "inflation", "monetarycondition", "monetarypolicy", "CDS", "usdcnh", "spread"],
        "valuation": ["pe", "pb", "AIAE"],
        "option": ["basis", "vol", "skew", "pcr"],
        "flow": ["margin", "amt", "sc_amt_pct", "idy_amt_corr", "idy_turn_pct", "trend", "highlow", "std", "IC_basis", "ht"]
    },
    'target_type': 'next_rebalance_return',   # 先做最基础版本
    'min_train_size': 120,                    # 调仓点数，不是日数
    'l1_ratio_grid': [0.1, 0.3, 0.5, 0.7, 0.9],
    'alpha_grid': np.logspace(-4, 1, 50),
    'n_splits': 5,
    'position_mode': 'ternary',               # 三挡仓位
    'signal_smoothing': False,                # 第一版先关
}


BACKTEST_CONFIG = {
    'start_date': '2016-01-01',
    'end_date': '2026-04-03',
    'index_code': 'I000905',
    'exec_price_type': 'close',   # 'open' or 'close0' or 'close'
    'fee_rate': 0,
    'slippage_rate': 0,
    'rebalance_freq': 'weekly',   # 'daily', 'weekly', 'monthly'
    'rebalance_weekday': 4,       # 0=Monday, ..., 4=Friday
    'rebalance_monthday': -1
}

# ==========================================

In [3]:
# 构造组内数据层 
def load_group_feature_matrix(group_feature_names):
    """
    输入:
        ["growth", "inflation", ...]
    输出:
        DataFrame:
            index = date
            columns = feature names
    """
    series_list = []

    for feat in group_feature_names:
        file_name = f"{feat}_signal.xlsx"
        s = backtest.load_signal(file_name)
        s.name = feat
        series_list.append(s)

    X = pd.concat(series_list, axis=1).sort_index()
    X = X.replace([np.inf, -np.inf], np.nan)
    return X

In [4]:
# 构造调仓级训练表
def build_rebalance_dataset(config, feature_names):
    close_df = backtest.load_index_price(config['index_code'], 'close').rename(columns={'price': 'close_price'})
    X = load_group_feature_matrix(feature_names)

    df = pd.concat([X, close_df], axis=1).sort_index()
    df = df.dropna(subset=['close_price'])

    trading_days = df.index.copy()

    rebalance_dates = backtest.generate_rebalance_dates(
        trading_days,
        freq=config['rebalance_freq'],
        weekday=config.get('rebalance_weekday', 0),
        monthday=config.get('rebalance_monthday', -1)
    )

    rebalance_dates = rebalance_dates.intersection(df.index)
    rb = df.loc[rebalance_dates].copy()

    rb['target_return'] = rb['close_price'].shift(-1) / rb['close_price'] - 1
    rb = rb.dropna(subset=['target_return'])

    return rb

In [5]:
def load_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"文件不存在: {file_path}")
    _, ext = os.path.splitext(file_path)
    if ext.lower() == '.csv':
        return pd.read_csv(file_path)
    elif ext.lower() in ('.xlsx', '.xls'):
        return pd.read_excel(file_path)
    elif ext.lower() == '.pkl':
        return pd.read_pickle(file_path)
    else:
        raise ValueError(f"不支持的格式: {ext}")


def load_signal(signal_name):
    fp = os.path.join(SIGNAL_PATH, signal_name)
    df = load_data(fp)
    if df.shape[1] < 2:
        raise ValueError("信号文件至少需两列：日期 + 信号")
    df = df.iloc[:, :2].copy()
    df.columns = ['date', 'signal']
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    df['signal'] = pd.to_numeric(df['signal'], errors='coerce')
    return df['signal'].dropna()

In [6]:
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import TimeSeriesSplit

def generate_group_enet_signal(config, group_name, feature_names, enet_config):
    rb = build_rebalance_dataset(config, feature_names)
    feature_cols = feature_names

    preds = pd.Series(index=rb.index, dtype=float)
    coef_records = []

    min_train_size = enet_config['min_train_size']

    for i in range(min_train_size, len(rb)):
        train = rb.iloc[:i].copy()
        test = rb.iloc[i:i+1].copy()

        X_train = train[feature_cols].copy()
        y_train = train['target_return'].copy()
        X_test = test[feature_cols].copy()

        med = X_train.median()
        X_train = X_train.fillna(med)
        X_test = X_test.fillna(med)

        mu = X_train.mean()
        sigma = X_train.std().replace(0, 1.0)

        X_train_z = (X_train - mu) / sigma
        X_test_z = (X_test - mu) / sigma

        n_splits = min(enet_config['n_splits'], max(2, len(X_train_z) // 20))
        tscv = TimeSeriesSplit(n_splits=n_splits)

        model = ElasticNetCV(
            l1_ratio=enet_config['l1_ratio_grid'],
            alphas=enet_config['alpha_grid'],
            cv=tscv,
            max_iter=20000,
            random_state=42
        )
        model.fit(X_train_z, y_train)

        pred = model.predict(X_test_z)[0]
        preds.iloc[i] = pred

        coef_row = {'date': test.index[0], 'intercept': model.intercept_}
        coef_row.update({col: coef for col, coef in zip(feature_cols, model.coef_)})
        coef_records.append(coef_row)

    coef_df = pd.DataFrame(coef_records).set_index('date') if coef_records else pd.DataFrame()
    preds = preds.dropna()
    preds.name = f'{group_name}_pred'

    return preds, coef_df
        

In [7]:
# 组内 prediction 转子信号
def normalize_group_signal(preds, window=60):
    mu = preds.rolling(window, min_periods=20).mean()
    sigma = preds.rolling(window, min_periods=20).std().replace(0, np.nan)
    z = (preds - mu) / sigma
    return z.clip(-3, 3)

# 组内先标准化成 z-score
def normalize_group_signal(preds, window=60):
    mu = preds.rolling(window, min_periods=20).mean()
    sigma = preds.rolling(window, min_periods=20).std().replace(0, np.nan)
    z = (preds - mu) / sigma
    return z.clip(-3, 3)

In [8]:
# 组间等权
def combine_group_signals_equal_weight(group_signal_dict):
    """
    输入:
        {
            'macro': series,
            'valuation': series,
            'option': series,
            'flow': series
        }
    输出:
        combined_score
    """
    aligned = pd.concat(group_signal_dict.values(), axis=1)
    aligned.columns = list(group_signal_dict.keys())
    combined = aligned.mean(axis=1)
    combined.name = 'combined_score'
    return combined, aligned

In [9]:
# 仓位函数
def map_score_to_position(score, window=60):
    """
    组合分数 -> 仓位
    """
    signal = pd.Series(index=score.index, dtype=float)

    for i in range(len(score)):
        hist = score.iloc[:i+1].dropna()
        if len(hist) < 20:
            signal.iloc[i] = 0.5
            continue

        q30 = hist.quantile(0.3)
        q70 = hist.quantile(0.7)
        val = score.iloc[i]

        if val >= q70:
            signal.iloc[i] = 1.0
        elif val <= q30:
            signal.iloc[i] = 0.0
        else:
            signal.iloc[i] = 0.5

    signal.name = 'signal'
    return signal

In [10]:
# 主函数
def run_enet_grouped_strategy():
    group_defs = ENET_CONFIG['groups']

    group_pred_dict = {}
    group_coef_dict = {}

    # 1) 逐组生成子信号
    for group_name, feature_names in group_defs.items():
        print(f"▶ 生成组内 ENET 子信号: {group_name}")
        preds, coef_df = generate_group_enet_signal(
            BACKTEST_CONFIG, group_name, feature_names, ENET_CONFIG
        )
        preds_norm = normalize_group_signal(preds, window=60)

        group_pred_dict[group_name] = preds_norm
        group_coef_dict[group_name] = coef_df

    # 2) 组间等权
    combined_score, group_signal_df = combine_group_signals_equal_weight(group_pred_dict)

    # 3) 总分数 -> 仓位信号
    final_signal = map_score_to_position(combined_score, window=60)

    # 4) 回测
    result = run_backtest_t_plus_1(final_signal, BACKTEST_CONFIG)
    perf_table = generate_performance_table(result)
    fig_paths = plot_backtest_charts(result, final_signal, BACKTEST_CONFIG, OUTPUT_PATH)

    # 5) 输出报表
    output_file = os.path.join(OUTPUT_PATH, "enet_grouped_equal_weight_report.xlsx")
    save_report_with_charts(result, perf_table, fig_paths, output_file)

    # 6) 保存信号明细
    detail_df = pd.concat([
        group_signal_df,
        combined_score,
        final_signal
    ], axis=1)
    detail_df.to_excel(os.path.join(OUTPUT_PATH, "enet_grouped_signal_detail.xlsx"))

    # 7) 保存各组系数
    coef_path = os.path.join(OUTPUT_PATH, "enet_group_coefficients.xlsx")
    with pd.ExcelWriter(coef_path) as writer:
        for group_name, coef_df in group_coef_dict.items():
            if not coef_df.empty:
                coef_df.to_excel(writer, sheet_name=group_name[:31])

    print(f"✅ 完成 ENET 组内 + 等权组间策略: {output_file}")

In [11]:
run_enet_grouped_strategy()

▶ 生成组内 ENET 子信号: macro


ValueError: Input contains NaN, infinity or a value too large for dtype('float64').